# 6. A sample for the language model, and notes on authors from Wikidata

The language model (notebook 7) is too slow for all 163,169 records on free hardware, so I let it classify a sample of 5,000,
stratified by the lexicon class of notebook 3: 1,250 records the lexicon marks spiritual, 1,250 other pastoral and 2,500 unmarked.
Each record carries a weight (records in its stratum ÷ records sampled), so I can estimate totals for the whole catalog.

I also add, where I can, a short note on the author from Wikidata (description, occupation, religious order). Many authors of one
or two books are not in Wikidata, and the pilot in notebook 7 showed that the note does not help, but I keep it so the pilot can
be repeated.

**Linking authors to Wikidata.** The BNE authority file for persons (`persona`) gives the Wikidata identifier of about 150,000
people. I look up a record's first author by its full heading ("Puente, Luis de la, ( 1554-1624)"); USTC headings have no dates,
so I look them up by name alone, and only when the name has a single Wikidata identifier. I drop people born after 1790 (wrong
homonyms, such as a film actor).

---

# 6. Una muestra para el modelo de lenguaje, y notas sobre los autores desde Wikidata

El modelo de lenguaje (notebook 7) es demasiado lento para los 163.169 registros con hardware gratuito, así que le hago clasificar
una muestra de 5.000, estratificada por la clase del léxico del notebook 3: 1.250 registros que el léxico marca como espirituales,
1.250 como otra pastoral y 2.500 sin marca. Cada registro lleva un peso (registros de su estrato ÷ registros de la muestra), y con
eso estimo los totales del catálogo completo.

Además agrego, cuando puedo, una nota breve sobre el autor tomada de Wikidata (descripción, ocupación, orden religiosa). Muchos
autores de uno o dos libros no están en Wikidata, y el piloto del notebook 7 mostró que la nota no ayuda, pero la conservo para que
el piloto se pueda repetir.

**Enlace de autores con Wikidata.** El archivo de autoridades de personas de la BNE (`persona`) trae el identificador de Wikidata
de unas 150.000 personas. Busco el primer autor de cada registro por su encabezamiento completo ("Puente, Luis de la, ( 1554-1624)");
los encabezamientos del USTC no tienen fechas, así que los busco solo por el nombre, y solo cuando ese nombre tiene un único
identificador de Wikidata. Descarto a las personas nacidas después de 1790 (homónimos equivocados, como un actor de cine).

In [1]:
# Celda 1
# Paths and parameters:
%cd "E:/No sincroniza/Software desarrollado/spanish-imprints-counts"
DATA = "E:/Datos/spanish_imprints"
CATALOG = DATA + "/catalog.tsv"                       # From notebook 3.
BNE_PERSONS = DATA + "/bne/persona-UTF8.zip"          # Downloaded by hand from datosabiertos.bne.es.
WIKIDATA_AUTHORS = DATA + "/wikidata_authors.tsv"     # Saved by this notebook; delete it to query Wikidata again.
SAMPLE = "data/llm_catalog_sample.tsv"
SAMPLE_SIZES = {"spiritual": 1250, "other_pastoral": 1250, "none": 2500}
SEED = 7
BORN_AFTER = 1790
# Extra records for the BNE sixteenth century, the smallest catalog-century cell of the sample (193 records):
EXTRA_SAMPLE = "data/llm_bne16_extra_sample.tsv"
EXTRA_SIZES = {"spiritual": 186, "other_pastoral": 226, "none": 395}

E:\No sincroniza\Software desarrollado\spanish-imprints-counts


In [2]:
# Celda 2
import os, re, time, zipfile, requests
import pandas as pd
from lexicon import fold
catalog = pd.read_csv(CATALOG, sep="\t", dtype={"id": str})
catalog["author_text"] = catalog["author_text"].fillna("")
print(catalog.shape)

(163169, 16)


## 1. Which authors can I link to Wikidata? / ¿Qué autores puedo enlazar con Wikidata?

In [3]:
# Celda 3
persona = pd.read_csv(zipfile.ZipFile(BNE_PERSONS).open("per/persona-UTF8.csv"), sep=";", dtype=str,
                      usecols=["nombre_de_persona", "otros_identificadores"], encoding_errors="replace")
persona["qid"] = persona["otros_identificadores"].str.extract(r"wikidata\.org/entity/(Q\d+)")[0]
persona["name"] = persona["nombre_de_persona"].fillna("").str.strip()
with_qid = persona.dropna(subset=["qid"])
print("Persons:", len(persona), "| with a Wikidata identifier:", len(with_qid))

Persons: 3316906 | with a Wikidata identifier: 151726


In [4]:
# Celda 4
# Full heading -> identifier, only when the heading has a single identifier:
qids_per_name = with_qid.groupby("name")["qid"].nunique()
single = with_qid[with_qid["name"].isin(qids_per_name[qids_per_name == 1].index)].drop_duplicates("name")
name_to_qid = dict(zip(single["name"], single["qid"]))

# "Puente, Luis de la, ( 1554-1624)" -> "puente luis de la":
def bare_name(name):
    return fold(re.split(r",?\s*\(", name)[0]).strip()

# Name without dates -> identifier, again only when there is a single one:
bare = pd.DataFrame({"bare": [bare_name(n) for n in name_to_qid], "qid": list(name_to_qid.values())})
qids_per_bare = bare.groupby("bare")["qid"].nunique()
single_bare = bare[bare["bare"].isin(qids_per_bare[qids_per_bare == 1].index)].drop_duplicates("bare")
bare_to_qid = dict(zip(single_bare["bare"], single_bare["qid"]))
print(len(name_to_qid), "headings and", len(bare_to_qid), "bare names with one identifier")

151717 headings and 147136 bare names with one identifier


In [5]:
# Celda 5
# First author of the record, without a trailing role: "Nieremberg, Juan Eusebio, ( 1595-1658) /**/ Bonet..." -> "Nieremberg, Juan Eusebio, ( 1595-1658)":
def first_author(author_text):
    first = author_text.split("/**/")[0].strip()
    return re.sub(r"\(\s*(?:autor|editor|traductor|impresor|librero|coautor|comentarista|prologuista)[^)]*\)\s*$", "", first).strip()

def find_qid(author_text):
    if not author_text:
        return None
    name = first_author(author_text)
    if name in name_to_qid:
        return name_to_qid[name]
    return bare_to_qid.get(bare_name(name))

catalog["qid"] = catalog["author_text"].map(find_qid)
with_author = catalog["author_text"] != ""
print("Records with an author:", with_author.sum(), "| linked to Wikidata:", catalog["qid"].notna().sum(),
      f"({catalog['qid'].notna().sum() / with_author.sum():.0%})")

Records with an author: 92580 | linked to Wikidata: 25807 (28%)


## 2. What does Wikidata say about them? / ¿Qué dice Wikidata de ellos?

In [6]:
# Celda 6
# One SPARQL query per 300 people: Spanish description, occupations (P106), religious orders (P611), fields (P101),
# genres (P136) and birth year (P569). Skipped when the file is already saved:
QUERY = """SELECT ?item ?description ?occupations ?orders ?fields ?genres ?born WHERE {
  VALUES ?item { %s }
  OPTIONAL { ?item schema:description ?description FILTER(LANG(?description) = "es") }
  OPTIONAL { SELECT ?item (GROUP_CONCAT(DISTINCT ?o; separator=", ") AS ?occupations) WHERE { VALUES ?item { %s } ?item wdt:P106/rdfs:label ?o FILTER(LANG(?o) = "es") } GROUP BY ?item }
  OPTIONAL { SELECT ?item (GROUP_CONCAT(DISTINCT ?r; separator=", ") AS ?orders) WHERE { VALUES ?item { %s } ?item wdt:P611/rdfs:label ?r FILTER(LANG(?r) = "es") } GROUP BY ?item }
  OPTIONAL { SELECT ?item (GROUP_CONCAT(DISTINCT ?f; separator=", ") AS ?fields) WHERE { VALUES ?item { %s } ?item wdt:P101/rdfs:label ?f FILTER(LANG(?f) = "es") } GROUP BY ?item }
  OPTIONAL { SELECT ?item (GROUP_CONCAT(DISTINCT ?g; separator=", ") AS ?genres) WHERE { VALUES ?item { %s } ?item wdt:P136/rdfs:label ?g FILTER(LANG(?g) = "es") } GROUP BY ?item }
  OPTIONAL { SELECT ?item (MIN(YEAR(?b)) AS ?born) WHERE { VALUES ?item { %s } ?item wdt:P569 ?b } GROUP BY ?item }
}"""

if not os.path.exists(WIKIDATA_AUTHORS):
    qids = sorted(catalog["qid"].dropna().unique())
    rows = []
    for start in range(0, len(qids), 300):
        values = " ".join("wd:" + q for q in qids[start:start + 300])
        reply = requests.post("https://query.wikidata.org/sparql", data={"query": QUERY % ((values,) * 6), "format": "json"},
                              headers={"User-Agent": "spanish-imprints-counts (research script)"}, timeout=120)
        for b in reply.json()["results"]["bindings"]:
            rows.append({k: v["value"] for k, v in b.items()})
        time.sleep(2)
    authors = pd.DataFrame(rows)
    authors["qid"] = authors["item"].str.rsplit("/", n=1).str[-1]
    authors = authors.drop(columns="item").drop_duplicates("qid")
    authors["born"] = pd.to_numeric(authors["born"], errors="coerce")
    authors = authors[~(authors["born"] > BORN_AFTER)]
    authors.to_csv(WIKIDATA_AUTHORS, sep="\t", index=False)
authors = pd.read_csv(WIKIDATA_AUTHORS, sep="\t", dtype=str)
print(len(authors), "authors")
authors.notna().mean().round(2)

2359 authors


genres         0.06
orders         0.22
occupations    0.92
description    0.78
fields         0.13
qid            1.00
born           0.94
dtype: float64

In [7]:
# Celda 7
# "religioso y misionero navarro. Ocupación: escritor, teólogo. Orden: Compañía de Jesús":
def note(row):
    parts = [row["description"]] if pd.notna(row["description"]) else []
    for label, column in [("Ocupación", "occupations"), ("Orden", "orders"), ("Campo", "fields"), ("Géneros", "genres")]:
        if pd.notna(row[column]):
            parts.append(f"{label}: {row[column]}")
    return ". ".join(parts)

authors["author_bio"] = authors.apply(note, axis=1)
catalog["author_bio"] = catalog["qid"].map(dict(zip(authors["qid"], authors["author_bio"]))).fillna("")
print("Records with a note on the author:", (catalog["author_bio"] != "").sum())
catalog.loc[catalog["author_bio"] != "", ["author", "author_bio"]].head()

Records with a note on the author: 24298


,author,author_bio
10,"Ortiz, Diego","Ocupación: compositor, musicólogo, teórico de ..."
12,"Fonseca, Cristóbal de","escritor español. Ocupación: escritor, filósof..."
17,"Pérez de Valdivia, Diego","sacerdote, teólogo y catedrático español. Ocup..."
22,"Núñez Delgadillo, Agustín",1570-1631
28,Thomas a Kempis,canónigo regular de San Agustín. Ocupación: es...


## 3. Which records go into the sample? / ¿Qué registros entran en la muestra?

In [8]:
# Celda 8
parts = []
for stratum, n in SAMPLE_SIZES.items():
    rows = catalog[catalog["lexicon_class"] == stratum].sample(n, random_state=SEED)
    rows["weight"] = (catalog["lexicon_class"] == stratum).sum() / n
    parts.append(rows)
sample = pd.concat(parts).sample(frac=1, random_state=SEED)
sample = sample.drop(columns="author").rename(columns={"author_text": "author"})
sample = sample[["source", "id", "author", "title", "author_bio", "lexicon_class", "weight", "year", "pages"]]
sample.to_csv(SAMPLE, sep="\t", index=False)
sample.groupby("lexicon_class").agg(records=("id", "size"), weight=("weight", "first"),
                                    with_note=("author_bio", lambda b: round((b != "").mean(), 2)))

,records,weight,with_note
lexicon_class,,,
none,2500,52.8272,0.14
other_pastoral,1250,16.5360,0.17
spiritual,1250,8.3448,0.23


## 4. An extra sample for the BNE sixteenth century / Una muestra extra para la BNE del siglo XVI

In notebook 8 I estimate each catalog and century separately. The BNE sixteenth century has only 193 records in the sample, so I add 807 more, drawn at random from the BNE records of 1501–1600 that are not yet in the sample, within each lexicon class. With them the cell reaches 250 spiritual, 250 other pastoral and 500 unmarked records.

En el notebook 8 estimo cada catálogo y cada siglo por separado. La BNE del siglo XVI tiene solo 193 registros en la muestra, así que le sumo 807 más, sacados al azar, dentro de cada clase del léxico, de los registros BNE de 1501–1600 que todavía no están en la muestra. Con ellos la celda llega a 250 registros espirituales, 250 de otro pastoral y 500 sin marca.


In [9]:
# Celda 9
# BNE records of 1501-1600 not in the sample yet, by lexicon class:
in_sample = set(sample["source"] + "|" + sample["id"])
bne16 = catalog[(catalog["source"] == "bne") & (catalog["year"] <= 1600)]
bne16 = bne16[~(bne16["source"] + "|" + bne16["id"]).isin(in_sample)]
parts = []
for stratum, n in EXTRA_SIZES.items():
    parts.append(bne16[bne16["lexicon_class"] == stratum].sample(n, random_state=SEED))
extra = pd.concat(parts).sample(frac=1, random_state=SEED)
extra = extra.drop(columns="author").rename(columns={"author_text": "author"})
extra = extra[["source", "id", "author", "title", "author_bio", "lexicon_class", "year", "pages"]]
extra.to_csv(EXTRA_SAMPLE, sep="	", index=False)
extra["lexicon_class"].value_counts()


none              395
other_pastoral    226
spiritual         186
Name: lexicon_class, dtype: int64